<img src="https://data-analytics-fullstack-assets.s3.eu-west-3.amazonaws.com/M05-Exploratory_Data_Analysis/M3_D2_Streamforge.png" />

# StreamForge engagement report 📺

You work as a junior data analyst at **StreamForge**, a streaming platform that operates in seven countries.

The growth team needs a weekly engagement report. They want to know which titles users actually watch, how engagement changes by country and by genre, and where the platform has the most playback errors. The product team will use this report to decide which shows to promote next month.

The data lives in three CSV exports. Each export has its own gaps and quirks. Your job is to combine the files, deal with the missing values, and produce the answers the growth team needs.

## Files you will use

- `StreamForge_viewing_events.csv`: one row per playback event (1,000 rows). Each row records what a user did on a title at a given time.
- `StreamForge_user_profiles.csv`: one row per user (50 rows). Plan, age, country, and monthly revenue.
- `StreamForge_content.csv`: one row per title (100 rows). Genre, content type, and rating.

<Note type="important">

Download the three CSV files from the resources section before you start. Place them in the same folder as this notebook.

</Note>

## Setup

Run the cell below to import Pandas and NumPy. You will use these two libraries through the whole exercise.

In [2]:
# import libraries
import pandas as pd
import numpy as np

## Task 1: Load the three files

Before you can analyze anything, you need to read the data into memory. The growth team wants an honest look at the source files. They do not want you to clean anything yet. Loading the files first lets you see what you are working with.

Read each of the three CSV files into its own DataFrame. Name them `events`, `users`, and `content`. Then print the shape of each DataFrame and look at the first five rows. Confirm that the column names match what you expect from the file descriptions above.

You should end up with three separate DataFrames sitting in memory, ready for analysis.

In [3]:
# Load the data
events = pd.read_csv("src/StreamForge_viewing_events.csv")
users = pd.read_csv("src/StreamForge_user_profiles.csv")
content = pd.read_csv("src/StreamForge_content.csv")

# print the shapes of the dataframes
print("Events shape:", events.shape)
print("Users shape:", users.shape)
print("Content shape:", content.shape)


Events shape: (1000, 11)
Users shape: (50, 5)
Content shape: (100, 5)


In [4]:
# Look at the first few rows of events
events.head()


,user_id,timestamp,event_type,duration_seconds,watched_duration,total_duration,content_id,episode_number,watched_seconds,total_seconds,user_rating
0,user_041,2024-02-05 07:14:00,play,1443,1186,1862,content_015,1,1186,1862.0,3.7
1,user_035,2024-02-24 01:01:00,error,251,80,1203,content_012,1,80,1203.0,3.0
2,user_002,2024-03-24 22:34:00,pause,919,316,2758,content_072,1,316,2758.0,NaN
3,user_001,2024-03-30 13:21:00,pause,629,141,1510,content_098,1,141,1510.0,NaN
4,user_022,2024-02-18 03:22:00,play,3117,2018,6422,content_014,1,2018,6422.0,1.2


In [5]:
# Look at the first few rows of users
users.head()

,user_id,subscription_type,user_age,country,monthly_revenue
0,user_001,premium,32.0,DE,13.99
1,user_002,standard,68.0,CA,8.99
2,user_003,premium,35.0,NaN,17.99
3,user_004,premium,56.0,JP,8.99
4,user_005,premium,21.0,DE,13.99


In [6]:
# Look at the first few rows of content
content.head()

,content_id,genre,primary_genre,content_type,rating
0,content_001,action,action,series,7.2
1,content_002,documentary,documentary,documentary,9.5
2,content_003,documentary,documentary,series,7.0
3,content_004,drama,drama,documentary,7.3
4,content_005,documentary,documentary,movie,7.1


## Task 2: Find the missing values

Real data is never complete. Some users skip the optional fields when they sign up. Some playback events lose data because of network problems. Before you compute anything, you need to know where the gaps are.

Use `isnull()` together with `sum()` to count the missing values in each column of each DataFrame. Print the result for the three DataFrames. Then write one short sentence in a Markdown cell that describes what you found. Mention which columns have gaps and roughly how many rows are affected.

<Note type="hint">

Calling `df.isnull()` returns a DataFrame of `True` and `False` values. Calling `.sum()` on that turns each column into a count of missing entries. So `df.isnull().sum()` gives you one number per column.

</Note>

In [7]:
# Missing values in events
missing_events = events.isnull().sum()
display(missing_events)


user_id               0
timestamp             0
event_type            0
duration_seconds      0
watched_duration      0
total_duration        0
content_id            0
episode_number        0
watched_seconds       0
total_seconds        50
user_rating         300
dtype: int64

In [8]:
# Missing values in users
missing_users = users.isnull().sum()
display(missing_users)


user_id              0
subscription_type    0
user_age             6
country              3
monthly_revenue      0
dtype: int64

In [9]:
# Missing values in content
missing_content = content.isnull().sum()
display(missing_content)

content_id       0
genre            0
primary_genre    0
content_type     0
rating           0
dtype: int64

**What I found:**
...

## Task 3: Decide what to do with each gap

Now you know where the missing values are. The next step is to decide how to handle each one. There is no single right answer. The decision depends on what you want to measure.

The growth team will ask you to compute completion rates and average watch time later. Read the rules below and apply each one to the right column.

- First rule: when `total_seconds` is missing in `events`, you cannot compute completion for that row. Drop those rows. Use `dropna()` with the `subset` parameter. Save the result as `events_clean`.

- Second rule: when `user_rating` is missing in `events`, the user simply did not rate the title. That is normal user behavior, not a data error. Leave the `NaN` values in place. Pandas aggregation functions like `mean()` ignore `NaN` automatically.

- Third rule: when `user_age` is missing in `users`, fill the gap with the median age. Median is safer than mean because it does not move when a few users have extreme ages. Use `fillna()`. Save the result as `users_clean`.

- Fourth rule: when `country` is missing in `users`, you cannot guess where they live. Leave the `NaN` in place for now. You will handle this case when you produce country-level reports.

After you apply the rules, print the shape of `events_clean` and `users_clean`. Confirm that the missing values are gone from the columns you cleaned and still present in the columns you left alone.

<Note type="hint">

`dropna(subset=["col_name"])` removes only the rows where `col_name` is missing. Other missing values stay.

`fillna(value)` replaces every `NaN` in the chosen column with the value you give it.

</Note>

In [10]:
# Rule 1: drop events without total_seconds
events_clean = events.dropna(subset=['total_seconds'])
display(events_clean.shape)


(950, 11)

In [11]:
# Rule 3: fill missing user_age with the median age
median_age = users['user_age'].median()
users_clean = users.copy()
users_clean['user_age'] = users_clean['user_age'].fillna(median_age)

# Create a copy of users to avoid modifying the original dataframe



In [12]:
# Confirm: missing values gone where we cleaned, still present where we kept them
print("events_clean missing values:")
display(events_clean.isnull().sum())


print()
print("users_clean missing values:")
display(users_clean.isnull().sum())

events_clean missing values:


user_id               0
timestamp             0
event_type            0
duration_seconds      0
watched_duration      0
total_duration        0
content_id            0
episode_number        0
watched_seconds       0
total_seconds         0
user_rating         290
dtype: int64


users_clean missing values:


user_id              0
subscription_type    0
user_age             0
country              3
monthly_revenue      0
dtype: int64

## Task 4: Summarize events by user, by content, and by event type

The growth team wants three quick numbers from the cleaned event data. Each one uses `groupby()` followed by `agg()`.

- First, find the top 10 users by total `watched_seconds`. Group `events_clean` by `user_id`, sum the `watched_seconds` column, and sort the result. The growth team uses this list to decide who receives the loyalty offer.

- Second, find the top 10 titles by total `watched_seconds`. For each title also report the number of sessions it had and the average `user_rating`. You need three aggregations on the same group, so use the `agg()` form that takes a dictionary or named keyword arguments. Sort by total watched time. The product team uses this list to plan promotions.

- Third, count how many events of each `event_type` happened on the platform. The engineering team uses this to track playback errors and buffer events.

Print all three results.

<Note type="hint">

When you need several aggregations at once, you can name each output column directly. For example:

```python
summary = events_clean.groupby("content_id").agg(
    total_watched=("watched_seconds", "sum"),
    session_count=("watched_seconds", "count"),
    avg_rating=("user_rating", "mean"),
)
```

This gives you one row per content with three named columns. The `mean` will silently skip any `NaN` in `user_rating`, which is what you want here.

</Note>

In [13]:
sum_watched_by_user = events_clean.groupby('user_id')['watched_seconds'].sum()
display(sum_watched_by_user)

user_id
user_001    16861
user_002     8707
user_003    18618
user_004    12664
user_005    15935
user_006    25136
user_007    15415
user_008    10034
user_009    11673
user_010     6652
user_011    12130
user_012    15435
user_013    10814
user_014     8560
user_015    14201
user_016    10473
user_017    12697
user_018    17407
user_019    13332
user_020    16241
user_021    14179
user_022    21055
user_023    10580
user_024    15964
user_025     7625
user_026    12283
user_027    16428
user_028    14902
user_029    12539
user_030    19436
user_031    20013
user_032    13321
user_033    16707
user_034    22472
user_035    14441
user_036    22668
user_037    29799
user_038    12294
user_039    10777
user_040    18765
user_041    13945
user_042    14557
user_043    15085
user_044    19337
user_045    13999
user_046    38534
user_047    25192
user_048    12421
user_049    17155
user_050    23319
Name: watched_seconds, dtype: int64

In [14]:
sorted_sumwatched_by_user = sum_watched_by_user.sort_values(ascending=False)
display(sorted_sumwatched_by_user)

user_id
user_046    38534
user_037    29799
user_047    25192
user_006    25136
user_050    23319
user_036    22668
user_034    22472
user_022    21055
user_031    20013
user_030    19436
user_044    19337
user_040    18765
user_003    18618
user_018    17407
user_049    17155
user_001    16861
user_033    16707
user_027    16428
user_020    16241
user_024    15964
user_005    15935
user_012    15435
user_007    15415
user_043    15085
user_028    14902
user_042    14557
user_035    14441
user_015    14201
user_021    14179
user_045    13999
user_041    13945
user_019    13332
user_032    13321
user_017    12697
user_004    12664
user_029    12539
user_048    12421
user_038    12294
user_026    12283
user_011    12130
user_009    11673
user_013    10814
user_039    10777
user_023    10580
user_016    10473
user_008    10034
user_002     8707
user_014     8560
user_025     7625
user_010     6652
Name: watched_seconds, dtype: int64

In [15]:
display(events_clean)

,user_id,timestamp,event_type,duration_seconds,watched_duration,total_duration,content_id,episode_number,watched_seconds,total_seconds,user_rating
0,user_041,2024-02-05 07:14:00,play,1443,1186,1862,content_015,1,1186,1862.0,3.7
1,user_035,2024-02-24 01:01:00,error,251,80,1203,content_012,1,80,1203.0,3.0
2,user_002,2024-03-24 22:34:00,pause,919,316,2758,content_072,1,316,2758.0,NaN
3,user_001,2024-03-30 13:21:00,pause,629,141,1510,content_098,1,141,1510.0,NaN
4,user_022,2024-02-18 03:22:00,play,3117,2018,6422,content_014,1,2018,6422.0,1.2
...,...,...,...,...,...,...,...,...,...,...,...
995,user_046,2024-01-17 09:46:00,buffer,1516,950,3223,content_085,1,950,3223.0,4.3
996,user_044,2024-03-01 05:39:00,stop,484,192,3313,content_084,1,192,3313.0,NaN
997,user_043,2024-01-23 06:10:00,pause,1300,633,1713,content_062,1,633,1713.0,4.2
998,user_032,2024-01-27 17:03:00,pause,97,54,1165,content_039,1,54,1165.0,NaN


In [16]:
top_10_users = sorted_sumwatched_by_user.head(10)
display(top_10_users)

user_id
user_046    38534
user_037    29799
user_047    25192
user_006    25136
user_050    23319
user_036    22668
user_034    22472
user_022    21055
user_031    20013
user_030    19436
Name: watched_seconds, dtype: int64

In [17]:
# Top 10 users by total watched seconds
top_10_users = events_clean.groupby('user_id')['watched_seconds'].sum()
display(top_10_users)

user_id
user_001    16861
user_002     8707
user_003    18618
user_004    12664
user_005    15935
user_006    25136
user_007    15415
user_008    10034
user_009    11673
user_010     6652
user_011    12130
user_012    15435
user_013    10814
user_014     8560
user_015    14201
user_016    10473
user_017    12697
user_018    17407
user_019    13332
user_020    16241
user_021    14179
user_022    21055
user_023    10580
user_024    15964
user_025     7625
user_026    12283
user_027    16428
user_028    14902
user_029    12539
user_030    19436
user_031    20013
user_032    13321
user_033    16707
user_034    22472
user_035    14441
user_036    22668
user_037    29799
user_038    12294
user_039    10777
user_040    18765
user_041    13945
user_042    14557
user_043    15085
user_044    19337
user_045    13999
user_046    38534
user_047    25192
user_048    12421
user_049    17155
user_050    23319
Name: watched_seconds, dtype: int64

watched_seconds  content_id  user_rating

content_id                                           

content_014            22978          15     2.690000

content_061            20167          23     3.092857

content_047            17998           9     2.383333

content_029            17672          11     2.557143

content_037            16937          11     2.866667

content_065            16568          15     3.500000

content_084            15726          10     2.650000

content_009            15608          17     2.578571

content_035            15334           8     4.740000

content_019            15178          13     3.007692

In [18]:
# Top 10 titles with three aggregations
titles_winsec = events_clean.groupby("content_id").agg(
        total_watched=("watched_seconds", "sum"),
        session_count=("watched_seconds", "count"),
        avg_rating=("user_rating", "mean"),
    )

top_10_titles_winsec = titles_winsec.sort_values(by="total_watched", ascending=False).head(10)

display(top_10_titles_winsec)
# Sort by total_watched and print the top 10


,total_watched,session_count,avg_rating
content_id,,,
content_014,22978,15,2.690000
content_061,20167,23,3.092857
content_047,17998,9,2.383333
content_029,17672,11,2.557143
content_037,16937,11,2.866667
content_065,16568,15,3.500000
content_084,15726,10,2.650000
content_009,15608,17,2.578571
content_035,15334,8,4.740000


In [19]:
# Count of events per event_type
count_events_per_eventtype = events_clean.groupby('event_type').size().sort_values(ascending=False)
display(count_events_per_eventtype)


event_type
pause     197
stop      194
error     189
buffer    186
play      184
dtype: int64

## Task 5: Enrich the events with user and content data

The summaries above only use the event file. To answer the growth team's real questions, you need to know which country each event came from and which genre each title belongs to. That information sits in the other two files.

Use `merge()` twice to bring the extra columns onto each event row.

- First, merge `events_clean` with `users_clean` on `user_id`. Use a left join so you keep every event even if a user record is missing. Save the result as `events_with_users`.

- Second, merge `events_with_users` with `content` on `content_id`. Again use a left join. Save the result as `events_full`.

Print the shape of `events_full` and the list of its columns. Confirm that you now have country, subscription type, primary_genre, and content_type sitting next to each event.

<Note type="hint">

A left join keeps all rows from the left DataFrame. Rows from the right DataFrame are matched on the join key. When there is no match, Pandas fills the new columns with `NaN`. The pattern is:

```python
merged = left_df.merge(right_df, on="key_column", how="left")
```

</Note>

In [20]:
# First merge: events + users
events_with_users = events_clean.merge(users_clean, on="user_id", how="left")
display(events_with_users)

,user_id,timestamp,event_type,duration_seconds,watched_duration,total_duration,content_id,episode_number,watched_seconds,total_seconds,user_rating,subscription_type,user_age,country,monthly_revenue
0,user_041,2024-02-05 07:14:00,play,1443,1186,1862,content_015,1,1186,1862.0,3.7,premium,26.0,FR,17.99
1,user_035,2024-02-24 01:01:00,error,251,80,1203,content_012,1,80,1203.0,3.0,standard,46.0,JP,8.99
2,user_002,2024-03-24 22:34:00,pause,919,316,2758,content_072,1,316,2758.0,NaN,standard,68.0,CA,8.99
3,user_001,2024-03-30 13:21:00,pause,629,141,1510,content_098,1,141,1510.0,NaN,premium,32.0,DE,13.99
4,user_022,2024-02-18 03:22:00,play,3117,2018,6422,content_014,1,2018,6422.0,1.2,standard,44.0,DE,13.99
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
945,user_046,2024-01-17 09:46:00,buffer,1516,950,3223,content_085,1,950,3223.0,4.3,standard,46.0,DE,17.99
946,user_044,2024-03-01 05:39:00,stop,484,192,3313,content_084,1,192,3313.0,NaN,premium,28.0,US,8.99
947,user_043,2024-01-23 06:10:00,pause,1300,633,1713,content_062,1,633,1713.0,4.2,premium,37.0,FR,13.99
948,user_032,2024-01-27 17:03:00,pause,97,54,1165,content_039,1,54,1165.0,NaN,standard,64.0,JP,13.99


In [21]:
# Second merge: add content metadata
events_full = events_with_users.merge(content, on="content_id", how="left")
display(events_full)

# List columns in events_full
print(events_full.shape)



,user_id,timestamp,event_type,duration_seconds,watched_duration,total_duration,content_id,episode_number,watched_seconds,total_seconds,user_rating,subscription_type,user_age,country,monthly_revenue,genre,primary_genre,content_type,rating
0,user_041,2024-02-05 07:14:00,play,1443,1186,1862,content_015,1,1186,1862.0,3.7,premium,26.0,FR,17.99,drama,drama,movie,7.8
1,user_035,2024-02-24 01:01:00,error,251,80,1203,content_012,1,80,1203.0,3.0,standard,46.0,JP,8.99,sci-fi,sci-fi,series,8.0
2,user_002,2024-03-24 22:34:00,pause,919,316,2758,content_072,1,316,2758.0,NaN,standard,68.0,CA,8.99,horror,horror,documentary,9.4
3,user_001,2024-03-30 13:21:00,pause,629,141,1510,content_098,1,141,1510.0,NaN,premium,32.0,DE,13.99,horror,horror,documentary,6.8
4,user_022,2024-02-18 03:22:00,play,3117,2018,6422,content_014,1,2018,6422.0,1.2,standard,44.0,DE,13.99,thriller,thriller,documentary,9.4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
945,user_046,2024-01-17 09:46:00,buffer,1516,950,3223,content_085,1,950,3223.0,4.3,standard,46.0,DE,17.99,romance,romance,movie,9.2
946,user_044,2024-03-01 05:39:00,stop,484,192,3313,content_084,1,192,3313.0,NaN,premium,28.0,US,8.99,romance,romance,movie,7.6
947,user_043,2024-01-23 06:10:00,pause,1300,633,1713,content_062,1,633,1713.0,4.2,premium,37.0,FR,13.99,romance,romance,series,7.9
948,user_032,2024-01-27 17:03:00,pause,97,54,1165,content_039,1,54,1165.0,NaN,standard,64.0,JP,13.99,horror,horror,documentary,8.3


(950, 19)


In [22]:
# Look at the first few rows of events_full
display(events_full.head(10))
print(f"nb colonne: {events_full.columns.to_list()}")

,user_id,timestamp,event_type,duration_seconds,watched_duration,total_duration,content_id,episode_number,watched_seconds,total_seconds,user_rating,subscription_type,user_age,country,monthly_revenue,genre,primary_genre,content_type,rating
0,user_041,2024-02-05 07:14:00,play,1443,1186,1862,content_015,1,1186,1862.0,3.7,premium,26.0,FR,17.99,drama,drama,movie,7.8
1,user_035,2024-02-24 01:01:00,error,251,80,1203,content_012,1,80,1203.0,3.0,standard,46.0,JP,8.99,sci-fi,sci-fi,series,8.0
2,user_002,2024-03-24 22:34:00,pause,919,316,2758,content_072,1,316,2758.0,NaN,standard,68.0,CA,8.99,horror,horror,documentary,9.4
3,user_001,2024-03-30 13:21:00,pause,629,141,1510,content_098,1,141,1510.0,NaN,premium,32.0,DE,13.99,horror,horror,documentary,6.8
4,user_022,2024-02-18 03:22:00,play,3117,2018,6422,content_014,1,2018,6422.0,1.2,standard,44.0,DE,13.99,thriller,thriller,documentary,9.4
5,user_030,2024-02-18 02:35:00,play,2701,2508,5275,content_069,1,2508,5275.0,3.5,premium,51.0,NaN,13.99,drama,drama,movie,7.9
6,user_024,2024-03-31 02:02:00,pause,1414,374,4580,content_074,1,374,4580.0,2.2,basic,46.0,UK,8.99,thriller,thriller,series,8.1
7,user_006,2024-02-18 08:29:00,play,5507,5068,7001,content_030,1,5068,7001.0,1.7,basic,24.0,BR,17.99,thriller,thriller,series,8.7
8,user_023,2024-03-30 21:41:00,stop,206,63,2393,content_027,1,63,2393.0,3.9,premium,68.0,JP,13.99,thriller,thriller,movie,6.4
9,user_011,2024-02-04 20:44:00,buffer,1200,344,4004,content_060,1,344,4004.0,2.3,premium,46.0,BR,17.99,documentary,documentary,series,7.8


nb colonne: ['user_id', 'timestamp', 'event_type', 'duration_seconds', 'watched_duration', 'total_duration', 'content_id', 'episode_number', 'watched_seconds', 'total_seconds', 'user_rating', 'subscription_type', 'user_age', 'country', 'monthly_revenue', 'genre', 'primary_genre', 'content_type', 'rating']


## Task 6: Compare metrics across two dimensions

Now that every event carries the user and content attributes, you can answer questions that mix two dimensions. The growth team wants three multi-level views.

- First, total `watched_seconds` by country and by `primary_genre`. This shows which genres each country prefers. Sort the result and print the top 10 country and genre combinations.

- Second, average `user_rating` by `subscription_type` and by `content_type`. This shows whether premium users rate movies differently from basic users. Round the result to two decimals so it reads cleanly.
    
Third, count of events by `country` and by `event_type`. This shows where errors and buffer events concentrate. The engineering team needs this view to find the worst regions for streaming quality.

When you group by `country`, remember that some users have a missing country. Pandas will create a separate group for `NaN` rows. Decide whether you want to keep that group or drop it. Use `dropna(subset=["country"])` before the groupby if you decide to drop it.

<Note type="hint">

Pass a list of columns to `groupby()` to group by more than one dimension at a time:

```python
result = events_full.groupby(["country", "primary_genre"])["watched_seconds"].sum()
```

The output has a MultiIndex with one level per grouping column.

</Note>

In [ ]:
# Watched seconds by country and primary_genre
# Drop rows with missing country before grouping
missing_country = events_full.isnull().sum()
display(missing_country)
event_full_cleand = events_full.dropna(subset=['country'])
# Group by country and primary_genre, sum watched_seconds
watched_seconds_by_country_genre = event_full_cleand.groupby(['country', 'primary_genre'])['watched_seconds'].sum()
display(watched_seconds_by_country_genre)

user_id                0
timestamp              0
event_type             0
duration_seconds       0
watched_duration       0
total_duration         0
content_id             0
episode_number         0
watched_seconds        0
total_seconds          0
user_rating          290
subscription_type      0
user_age               0
country               57
monthly_revenue        0
genre                  0
primary_genre          0
content_type           0
rating                 0
dtype: int64

country  primary_genre
BR       action            9775
         comedy           12242
         documentary      16289
         drama            24083
         horror            8986
         romance          19403
         sci-fi           42238
         thriller         42926
CA       action            6061
         comedy            6938
         documentary       3875
         drama             3002
         horror            9523
         romance           2836
         sci-fi           15557
         thriller         21330
DE       action            7667
         comedy            8178
         documentary      10516
         drama             6874
         horror            9673
         romance          27133
         sci-fi           18301
         thriller         14516
FR       action            8256
         comedy            7873
         documentary       4525
         drama             6425
         horror            5994
         romance          11038
         sci-fi  

In [24]:
# Average user_rating by subscription_type and content_type
average_user_rating = event_full_cleand.groupby(['subscription_type', 'content_type'])['user_rating'].mean().round(2)
print(average_user_rating)




subscription_type  content_type
basic              documentary     2.90
                   movie           2.99
                   series          2.70
premium            documentary     3.01
                   movie           3.12
                   series          3.16
standard           documentary     2.90
                   movie           3.18
                   series          2.91
Name: user_rating, dtype: float64


In [25]:
display(event_full_cleand)

,user_id,timestamp,event_type,duration_seconds,watched_duration,total_duration,content_id,episode_number,watched_seconds,total_seconds,user_rating,subscription_type,user_age,country,monthly_revenue,genre,primary_genre,content_type,rating
0,user_041,2024-02-05 07:14:00,play,1443,1186,1862,content_015,1,1186,1862.0,3.7,premium,26.0,FR,17.99,drama,drama,movie,7.8
1,user_035,2024-02-24 01:01:00,error,251,80,1203,content_012,1,80,1203.0,3.0,standard,46.0,JP,8.99,sci-fi,sci-fi,series,8.0
2,user_002,2024-03-24 22:34:00,pause,919,316,2758,content_072,1,316,2758.0,NaN,standard,68.0,CA,8.99,horror,horror,documentary,9.4
3,user_001,2024-03-30 13:21:00,pause,629,141,1510,content_098,1,141,1510.0,NaN,premium,32.0,DE,13.99,horror,horror,documentary,6.8
4,user_022,2024-02-18 03:22:00,play,3117,2018,6422,content_014,1,2018,6422.0,1.2,standard,44.0,DE,13.99,thriller,thriller,documentary,9.4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
945,user_046,2024-01-17 09:46:00,buffer,1516,950,3223,content_085,1,950,3223.0,4.3,standard,46.0,DE,17.99,romance,romance,movie,9.2
946,user_044,2024-03-01 05:39:00,stop,484,192,3313,content_084,1,192,3313.0,NaN,premium,28.0,US,8.99,romance,romance,movie,7.6
947,user_043,2024-01-23 06:10:00,pause,1300,633,1713,content_062,1,633,1713.0,4.2,premium,37.0,FR,13.99,romance,romance,series,7.9
948,user_032,2024-01-27 17:03:00,pause,97,54,1165,content_039,1,54,1165.0,NaN,standard,64.0,JP,13.99,horror,horror,documentary,8.3


In [26]:
# Event counts by country and event_type
event_count = event_full_cleand.groupby(['country', 'event_type']).size()

print(event_count.head(10))

country  event_type
BR       buffer        35
         error         50
         pause         47
         play          40
         stop          39
CA       buffer        10
         error         16
         pause         18
         play          15
         stop          23
dtype: int64
